# Fake Jobs – Experiment 1: TabPFN-unsupervised & FoMo-0D
- TabPFN-Unsupervised (Kontext <= 3000 aus Train, ohne Labels)
- FoMo-0D zero-shot (Features → 100, Kontext <= 5000 aus Train)
- Beide Kontexte werden aus **einem** `rng` gezogen; die Zellreihenfolge ist deshalb relevant

In [ ]:
import os
import sys
import glob
import zipfile
import time
import numpy as np
import pandas as pd
import torch
import mlflow
from sklearn.metrics import roc_auc_score, classification_report, precision_recall_curve, auc
from sklearn.preprocessing import QuantileTransformer
from tabpfn import TabPFNClassifier, TabPFNRegressor
from tabpfn_extensions.unsupervised import TabPFNUnsupervisedModel

SEED = int(os.environ.get("SEED", 1))
print("SEED", SEED)

## Daten & Split laden

In [ ]:
df = pd.read_csv(f"../../data/preprocessed/cleaned_fake_jobs_seed{SEED}.csv")
split = pd.read_csv(f"../../data/splits/split_fake_jobs_seed{SEED}.csv")
y = (df["fraudulent"]).values
X = df.drop(columns=["row_id", "fraudulent"]).values.astype(np.float32)

s = df["row_id"].map(split.set_index("row_id")["split"]).values
X_train = X[s == "train"]
X_test, y_test = X[s == "test"], y[s == "test"]
k = round(len(y_test) * y[s == "train"].mean())
print("train", X_train.shape, "test", X_test.shape, "| Outlier-Rate Test:", round(y_test.mean(), 4))

mlflow.set_tracking_uri("file:../../mlruns")
mlflow.set_experiment("fake_jobs_experiment_1")

## TabPFN-Unsupervised
- Kontext = zufällige Train-Teilmenge <= 3000 (ohne Labels)
- Score = `outliers()` in nativer Orientierung (höher = anomaler), kein labelbasierter Flip

In [ ]:
rng = np.random.RandomState(SEED)
idx = rng.choice(len(X_train), size=min(3000, len(X_train)), replace=False)
X_ctx = X_train[idx]

model = TabPFNUnsupervisedModel(tabpfn_clf=TabPFNClassifier(), tabpfn_reg=TabPFNRegressor())

t0 = time.perf_counter()
model.fit(torch.from_numpy(X_ctx))
scores = np.asarray(model.outliers(torch.from_numpy(X_test))).ravel().astype(float)
runtime = time.perf_counter() - t0

prec, rec, _ = precision_recall_curve(y_test, scores)
auprc = auc(rec, prec)
auroc = roc_auc_score(y_test, scores)
pred = (scores >= np.sort(scores)[-k]).astype(int)

with mlflow.start_run(run_name="tabpfn_unsup"):
    mlflow.log_params({"context_size": len(X_ctx), "seed": SEED})
    mlflow.log_metric("auprc", auprc)
    mlflow.log_metric("auc_roc", auroc)
    mlflow.log_metric("runtime_s", runtime)
print(f"tabpfn_unsup: AUPRC={auprc:.4f} AUC-ROC={auroc:.4f} time={runtime:.1f}s")
print(classification_report(y_test, pred, target_names=["inlier", "outlier"], digits=4, zero_division=0))

## FoMo-0D laden
- `FoMo0DHub` aus `../../FoMo-0D`, Gewichte aus `ckpt.zip`

In [ ]:
sys.path.insert(0, "../../FoMo-0D")
from fomo_hub import FoMo0DHub

ckpts = glob.glob("../../FoMo-0D/ckpt/**/best.ckpt", recursive=True)
if not ckpts:
    with zipfile.ZipFile("../../FoMo-0D/ckpt.zip") as z:
        z.extractall("../../FoMo-0D")
    ckpts = glob.glob("../../FoMo-0D/ckpt/**/best.ckpt", recursive=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
fomo = FoMo0DHub(num_features=100, emsize=256, nhid=512, nlayers=4, nhead=4, num_R=500)
state = torch.load(ckpts[0], map_location=device)["state_dict"]
state = {key.replace("model.", "", 1): v for key, v in state.items()}
missing, unexpected = fomo.model.load_state_dict(state, strict=False)
print("missing:", len(missing), "unexpected:", len(unexpected))
fomo = fomo.to(device).eval()

## FoMo-0D (zero-shot)
- Kontext = Train-Subsample <= 5000 (Originalverteilung); QuantileTransform nur auf dem Kontext gefittet
- Features per Zero-Padding auf genau 100; Score = Softmax-P(Klasse 1)

In [ ]:
cidx = rng.choice(len(X_train), size=min(5000, len(X_train)), replace=False)
X_ctx = X_train[cidx]

qt = QuantileTransformer(output_distribution="normal", random_state=SEED, n_quantiles=min(1000, len(X_ctx)))
X_ctx_t = qt.fit_transform(X_ctx)
X_test_t = qt.transform(X_test)
pad = 100 - X_ctx_t.shape[1]
X_ctx_p = np.pad(X_ctx_t, ((0, 0), (0, pad)), constant_values=0.0).astype(np.float32)
X_test_p = np.pad(X_test_t, ((0, 0), (0, pad)), constant_values=0.0).astype(np.float32)

train_x = torch.from_numpy(X_ctx_p).unsqueeze(1).to(device)  # (ctx, 1, 100)
t0 = time.perf_counter()
probs = []
with torch.no_grad():
    for start in range(0, len(X_test_p), 256):
        chunk = torch.from_numpy(X_test_p[start:start + 256]).unsqueeze(1).to(device)
        logits = fomo(train_x, chunk).squeeze(1)
        probs.append(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy())
scores = np.concatenate(probs).astype(float)
runtime = time.perf_counter() - t0

prec, rec, _ = precision_recall_curve(y_test, scores)
auprc = auc(rec, prec)
auroc = roc_auc_score(y_test, scores)
pred = (scores >= np.sort(scores)[-k]).astype(int)

with mlflow.start_run(run_name="fomo_od"):
    mlflow.log_params({"context_size": len(X_ctx), "seed": SEED})
    mlflow.log_metric("auprc", auprc)
    mlflow.log_metric("auc_roc", auroc)
    mlflow.log_metric("runtime_s", runtime)
print(f"fomo_od: AUPRC={auprc:.4f} AUC-ROC={auroc:.4f} time={runtime:.1f}s")
print(classification_report(y_test, pred, target_names=["inlier", "outlier"], digits=4, zero_division=0))